In [2]:
""" Cell 1 — 專案路徑與設定（Colab / 雲端資料夾版）"""
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/battle_cats_ml_test3")

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"找不到專案資料夾: {PROJECT_DIR}")

sys.path.insert(0, str(PROJECT_DIR))

!pip -q install pandas scikit-learn scipy

from game_labels import (
    ENEMY_TRAIT_CN,
    ABILITY_NAME_CN,
    MODULE_1_ABILITY_IDS,
    MODULE_2_ABILITY_IDS,
    MODULE_3_ABILITY_IDS,
    ALWAYS_ACTIVE_IDS,
)

JSON_SSR = PROJECT_DIR / "battlecats_ssr_db.json"
JSON_ALL = PROJECT_DIR / "battlecats_ALL_db.json"
JSON_PATH = JSON_SSR   # 預設 SSR

print("專案目錄:", PROJECT_DIR)
print("中文對照:", len(ENEMY_TRAIT_CN), "屬性,", len(ABILITY_NAME_CN), "能力")
print("模組一/二/三:", len(MODULE_1_ABILITY_IDS), len(MODULE_2_ABILITY_IDS), len(MODULE_3_ABILITY_IDS))
print("將讀取:", JSON_PATH.name, "存在?", JSON_PATH.is_file())

Mounted at /content/drive


FileNotFoundError: 找不到專案資料夾: /content/drive/MyDrive/battle_cats_ml_test3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
""" Cell 2 — 讀取 JSON """
import json

if not JSON_PATH.is_file():
    raise FileNotFoundError(f"找不到 {JSON_PATH}，請確認檔名與 Cell 1 的 JSON_PATH")

with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

n = len(raw) if isinstance(raw, (dict, list)) else 0
print("型別:", type(raw).__name__, "| 筆數:", n)

# 看第一筆的鍵（確認「排名」「稀有度」等欄位名）
first = next(iter(raw.values())) if isinstance(raw, dict) else raw[0]
print("單筆頂層欄位:", list(first.keys())[:15])

scores = []
for c in (raw.values() if isinstance(raw, dict) else raw):
    s = c.get("評分")
    if s not in (None, "", "NAN", "nan"):
        try:
            clipped_score = max(0.0, min(4.5, float(s)))
            scores.append(clipped_score)
        except (TypeError, ValueError):
            pass
print("有效評分筆數:", len(scores), "/", n)
if scores:
    print("評分範圍:", min(scores), "~", max(scores))

In [ ]:
""" Cell 3 — 特徵表 + 三模組公式分（0~10）"""
import importlib
import features
importlib.reload(features)
from features import build_feature_dataframe
df = build_feature_dataframe(raw)

import pandas as pd
from features import build_feature_dataframe

df = build_feature_dataframe(raw)
#print("列數:", len(df), "| 欄位數:", len(df.columns))
display(df[[
    "名字", "form_stage", "評分",
    "score_mod1", "score_mod2", "score_mod3",
    "mod1_panel_raw", "mod1_ability_raw", "mod1_raw",
]].head(10))

In [ ]:
""" Cell 4 — 標籤與三模組分 EDA """
labeled = df[df["評分"].notna()]
print("有評分:", len(labeled), "/", len(df))
display(labeled[["評分", "score_mod1", "score_mod2", "score_mod3"]].describe())

"""
print("\n模組1 與評分相關:", labeled["評分"].corr(labeled["score_mod1"]))
print("模組2 與評分相關:", labeled["評分"].corr(labeled["score_mod2"]))
print("模組3 與評分相關:", labeled["評分"].corr(labeled["score_mod3"]))
"""

In [ ]:
""" Cell 5 — 模組1 基礎數值"""
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

MOD1_FEATURES = [
    "score_mod1", "mod1_raw", "mod1_panel_raw", "mod1_ability_raw",
    "DPS", "體力", "射程", "速度", "攻擊頻率", "成本",
    "dps_per_cost", "hp_per_cost",
]
MOD1_FEATURES = [c for c in MOD1_FEATURES if c in df.columns]

labeled = df[df["評分"].notna()].copy()
X = labeled[MOD1_FEATURES].fillna(0)
y = labeled["評分"].astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model_m1 = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1,
)
model_m1.fit(X_tr, y_tr)
pred_te = model_m1.predict(X_te)
"""
print("模組1視角 — Spearman:", spearmanr(y_te, pred_te).statistic)
print("模組1視角 — MAE:", mean_absolute_error(y_te, pred_te))
"""

# 全表預測（含無評分角色）
df["pred_m1_評分"] = model_m1.predict(df[MOD1_FEATURES].fillna(0)).clip(0, 4.5)
df["score_10_pred_m1"] = 1.0 + (df["pred_m1_評分"] / 4.5) * 8.0
df["score_10_pred_m1_final"] = (df["score_10_pred_m1"] * 1.2).clip(0, 10.0)
display(df.sort_values("pred_m1_評分", ascending=False)[
    ["名字", "評分", "score_mod1", "pred_m1_評分", "score_10_pred_m1", "score_10_pred_m1_final"]
].head(15))

In [ ]:
""" Cell 6 — 模組2 針對屬性基礎數值 """
MOD2_FEATURES = [f"trait_raw_{i}" for i in range(15)]
MOD2_FEATURES = ["score_mod2"] + [c for c in MOD2_FEATURES if c in df.columns]

labeled = df[df["評分"].notna()].copy()
X = labeled[MOD2_FEATURES].fillna(0)
y = labeled["評分"].astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model_m2 = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1,
)
model_m2.fit(X_tr, y_tr)
pred_te = model_m2.predict(X_te)
"""
print("模組2視角 — Spearman:", spearmanr(y_te, pred_te).statistic)
print("模組2視角 — MAE:", mean_absolute_error(y_te, pred_te))
"""

# 全表預測（含無評分角色）
df["pred_m2_評分"] = model_m2.predict(df[MOD2_FEATURES].fillna(0)).clip(0, 4.5)
df["score_10_pred_m2"] = 1.0 + (df["pred_m2_評分"] / 4.5) * 8.0
df["score_10_pred_m2_final"] = (df["score_10_pred_m2"] * 1.2).clip(0, 10.0)
display(df.sort_values("pred_m2_評分", ascending=False)[
    ["名字", "評分", "針對屬性", "score_mod2", "pred_m2_評分", "score_10_pred_m2", "score_10_pred_m2_final"]
].head(15))

In [ ]:
""" Cell 7 — 模組3 針對屬性控場效果 """
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

# 定義模組3的特徵
# 包含 score_mod3 以及所有 control_raw_i 和 control_global_raw
MOD3_FEATURES = [f"control_raw_{i}" for i in range(15)] + ["control_global_raw"]
MOD3_FEATURES = ["score_mod3"] + [c for c in MOD3_FEATURES if c in df.columns]

# 篩選出有「評分」的資料
labeled = df[df["評分"].notna()].copy()
X = labeled[MOD3_FEATURES].fillna(0) # 填充缺失值
y = labeled["評分"].astype(float)

# 分割訓練集和測試集
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 訓練隨機森林迴歸模型
model_m3 = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1,
)
model_m3.fit(X_tr, y_tr)

# 在測試集上進行預測並評估
pred_te = model_m3.predict(X_te)
"""
print("模組3視角 — Spearman:", spearmanr(y_te, pred_te).statistic)
print("模組3視角 — MAE:", mean_absolute_error(y_te, pred_te))
"""

# 對整個 DataFrame 進行預測（包括沒有「評分」的角色）
df["pred_m3_評分"] = model_m3.predict(df[MOD3_FEATURES].fillna(0)).clip(0, 4.5)
df["score_10_pred_m3"] = 1.0 + (df["pred_m3_評分"] / 4.5) * 8.0
df["score_10_pred_m3_final"] = (df["score_10_pred_m3"] * 1.2).clip(0, 10.0)

# 顯示預測評分最高的15個角色
display(df.sort_values("pred_m3_評分", ascending=False)[
    ["名字", "評分", "針對屬性", "score_mod3", "pred_m3_評分", "score_10_pred_m3", "score_10_pred_m3_final"]
].head(15))

In [ ]:
""" Cell 8 — 搜尋 -> 顯示最相近10名 -> 選擇後顯示單一角色（完整顯示版） """
from difflib import get_close_matches
import pandas as pd
import re
from IPython.display import HTML, display
# ===== 顯示設定：避免內容被截斷 =====
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
# ===== 必要欄位檢查 =====
required_cols = [
    "名字",
    "score_10_pred_m1_final",
    "score_10_pred_m2_final",
    "score_10_pred_m3_final",
    "啟用能力_中文",
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"缺少欄位: {missing}，請先跑完 Cell 3 與 Cell 5~7。")
# ===== 能力關鍵字（對應 ID）=====
# 模組2條件：ID 1~5 任一存在才保留，否則顯示0
m2_keys = ["超大傷害", "極度傷害", "很耐打", "超級耐打", "善於攻擊"]
# 模組3條件：五種控場都要有，否則顯示0（ID 15/16/17/32/37）
m3_keys_all = ["打飛敵人", "使動作停止", "使動作變慢", "攻擊力下降", "詛咒"]
def has_any_m2(ability_text: str) -> bool:
    s = str(ability_text)
    return any(k in s for k in m2_keys)
def has_all_m3(ability_text: str) -> bool:
    s = str(ability_text)
    return all(k in s for k in m3_keys_all)
def unify_trait(row) -> str:
    """
    統一屬性顯示。優先使用「針對屬性」欄位，
    若格式為 '模組2/模組3'，取非'無'的一側。
    """
    if "針對屬性" in row.index and pd.notna(row["針對屬性"]):
        t = str(row["針對屬性"]).strip()
        if "/" in t:
            a, b = t.split("/", 1)
            a, b = a.strip(), b.strip()
            if a != "無":
                return a
            if b != "無":
                return b
            return "無"
        return t if t else "無"
    return "無"
def simplify_abilities(full_text: str) -> str:
    """
    原始:
      ID15 打飛敵人 → 紅色敵人 (機率50%); ID17 使動作變慢 → 紅色敵人 (機率50%)
    轉成:
      打飛敵人 → 機率50%
      使動作變慢 → 機率50%
    """
    s = str(full_text) if full_text is not None else ""
    if not s.strip():
        return ""
    parts = [p.strip() for p in s.split(";") if p.strip()]
    lines = []
    for p in parts:
        # 去掉前綴 IDxx
        p_no_id = re.sub(r"^ID\d+\s*", "", p).strip()
        # 能力名：箭頭前
        ability = p_no_id.split("→")[0].strip()
        # 機率：抓「機率xx%」或「xx%」
        m = re.search(r"機率\s*\d+%|\d+%", p_no_id)
        if m:
            prob_raw = m.group(0).replace("機率", "").strip()
            prob_txt = f"機率{prob_raw}"
        else:
            prob_txt = "機率100%"
        lines.append(f"{ability} → {prob_txt}")
    return "\n".join(lines)
# ===== Step 1: input 搜尋字 =====
query = input("請輸入角色名稱關鍵字：").strip()
if not query:
    raise ValueError("請輸入關鍵字。")
name_series = df["名字"].astype(str).fillna("")
# 先做包含比對
contains_names = name_series[name_series.str.contains(query, case=False, regex=False)].unique().tolist()
# 再補模糊比對，組成最多10個候選
fuzzy_names = get_close_matches(query, name_series.unique().tolist(), n=10, cutoff=0.3)
candidates = []
for n in contains_names + fuzzy_names:
    if n not in candidates:
        candidates.append(n)
    if len(candidates) >= 10:
        break
if not candidates:
    print(f"找不到與「{query}」相近的角色。")
else:
    print("\n最相近的 10 個名字：")
    for i, n in enumerate(candidates, start=1):
        print(f"{i}. {n}")
    # ===== Step 2: 選擇編號 =====
    choice = input(f"\n請輸入編號（1~{len(candidates)}）：").strip()
    if not choice.isdigit():
        raise ValueError("請輸入數字編號。")
    idx = int(choice)
    if idx < 1 or idx > len(candidates):
        raise ValueError(f"編號超出範圍，請輸入 1~{len(candidates)}。")
    target_name = candidates[idx - 1]
    # 若同名多列（不同型態/階段），全部顯示
    sel = df[df["名字"].astype(str) == target_name].copy()
    if sel.empty:
        raise ValueError("找不到所選角色資料。")
    # ===== 顯示欄位（不改原始df）=====
    sel["屬性"] = sel.apply(unify_trait, axis=1)
    sel["模組1分數"] = sel["score_10_pred_m1_final"]
    sel["模組2分數"] = sel["score_10_pred_m2_final"]
    sel["模組3分數"] = sel["score_10_pred_m3_final"]
    # 規則：僅 Cell 8 顯示時套用歸零
    m2_ok = sel["啟用能力_中文"].apply(has_any_m2)
    m3_ok = sel["啟用能力_中文"].apply(has_all_m3)
    sel.loc[~m2_ok, "模組2分數"] = 0.0
    sel.loc[~m3_ok, "模組3分數"] = 0.0
    # 啟用能力簡化 + 完整顯示（每條換行）
    sel["啟用能力"] = sel["啟用能力_中文"].apply(simplify_abilities)
    # 你指定的最終欄位
    show_cols = ["名字", "屬性", "模組1分數", "模組2分數", "模組3分數", "啟用能力"]
    out = sel[show_cols].copy()
    out["啟用能力"] = out["啟用能力"].str.replace("\n", "<br>", regex=False)
    print(f"\n你選的是：{target_name}")
    display(HTML(out.to_html(escape=False, index=False)))